# OOD testing: stand-in players that differ from their history

**Milestone:** "Out-of-distribution testing: measure how the player model behaves against
stand-in players whose style differs from their history, and against players much stronger or
weaker than estimated" (README Teaching). Also the README claim: *the engine never breaks on odd
moves; what can fail is the player model.*

Stand-in players are **engine personas**: an engine plus rules that give a style and a strength.
Because we control them, we know the ground truth the model should find.

## Experiment plan

| # | History | Test games | Expected model behaviour |
|---|---|---|---|
| E1 | persona A (N games) | persona A | estimate converges, confidence rises, OOD low |
| E2 | persona A | persona B (different style, same strength) | OOD rises; teacher falls back |
| E3 | persona A | A at +400 Elo (much stronger) | OOD rises fast; no exploitation of stale habits |
| E4 | persona A | A at -400 Elo (much weaker / rusty) | OOD rises; fallback at last estimated level |
| E5 | persona A | A "hiding strength" (plays weak early, strong later) | live OOD fires mid-game |

Measured per experiment: Elo estimate error, CI coverage, confidence, OOD score per game,
teacher fallback rate (`TeacherDecision.reason`), balanced share (metrics.ipynb).

**TODO checklist**
- [ ] Noisy-engine persona (`PersonaPlayer.play`).
- [ ] Game harness (`play_persona_game`) and model evaluation (`evaluate_model_on_games`).
- [ ] Self-checks (plan at the bottom).
- [ ] Style personas (`aggressive`, `passive`, `tactically blind`, `opening swapper`, `strength hider`).
- [ ] Strength anchors: Stockfish UCI_Elo personas, or human-like engines (Maia) for realism.
- [ ] Run E1..E5 in `run_teaching_experiment.ipynb` and record results in the Wiki.

In [ ]:
%run ./move_selection.ipynb

In [ ]:
import random
from dataclasses import dataclass, field
from typing import Optional

import chess
import chess.pgn

## Personas

In [ ]:
@dataclass
class Persona:
    name: str
    description: str
    target_elo: Optional[int] = None      # for UCI personas (Stockfish UCI_Elo)
    blunder_rate: float = 0.0             # chance of a uniformly random move
    style: str = "neutral"                # neutral | aggressive | passive | tactically_blind | ...


class PersonaPlayer:
    """An engine move most of the time, a random move with probability `blunder_rate`."""

    def __init__(self, persona: Persona, engine: EngineLike, seed: int = 0):
        """Store persona and engine; seeded random.Random for reproducible games."""
        self.persona, self.engine, self.rng = persona, engine, random.Random(seed)

    def play(self, board: chess.Board) -> chess.Move:
        """The persona's move."""
        # Design: styled personas delegate to style_move; neutral ones play a uniformly random
        # legal move with probability blunder_rate, else the engine's move. (Random moves are a
        # crude "mistake" model: real humans err plausibly, not uniformly - note it in results.)
        # TODO: implement
        raise NotImplementedError


def style_move(persona: Persona, board: chess.Board, engine: EngineLike, rng: random.Random) -> chess.Move:
    """Move for a styled persona."""
    # Design hints:
    # - aggressive: among multipv candidates within ~100 cp of the best, prefer captures, checks
    #   and pawn storms.
    # - passive: prefer quiet moves and trades.
    # - tactically_blind: never play the best move when it is forcing.
    # - strength_hider: blunder_rate high for ~15 moves, then 0.
    # TODO: implement
    raise NotImplementedError


PERSONAS = {
    "solid": Persona("solid", "engine moves, rare random blunder", blunder_rate=0.05),
    "sloppy": Persona("sloppy", "engine moves, frequent random blunders (rusty player)", blunder_rate=0.3),
    # TODO: styled personas, see style_move
    "aggressive": Persona("aggressive", "prefers captures and checks", style="aggressive"),
    "tactically_blind": Persona("tactically_blind", "misses forcing moves", style="tactically_blind"),
}

## Harness: play a persona against a policy (software only, no physical board)

In [ ]:
def play_persona_game(persona_player: PersonaPlayer, robot,  # any object with choose_move(board) (game_manager.MovePolicy)
                      persona_color: chess.Color = chess.WHITE,
                      max_plies: int = 200, game_id: str = "ood") -> chess.pgn.Game:
    """Play one game persona vs robot policy; return it as a PGN Game with headers."""
    # Design: alternate persona.play / robot.choose_move until game over (claim_draw) or max_plies;
    # chess.pgn.Game.from_board; set Event/Site to game_id and White/Black names so
    # habits.player_color_in_game can find the persona later.
    # TODO: implement
    raise NotImplementedError


def evaluate_model_on_games(model: PlayerModel, games: list, analyst: EngineLike, color: chess.Color,
                            true_elo: Optional[float] = None, update: bool = True) -> list[dict]:
    """For each game: OOD score vs the history *before* the game, then (optionally) update."""
    # Design:
    # - analyse_game -> summarize_game; compute ood_score *before* adding the game (otherwise the
    #   game is compared with a history that already contains it).
    # - update=False lets you test "different persona" games without polluting the history.
    # - One row per game: acpl, ood_score, Elo estimate + interval, confidence and, when the true
    #   Elo is known, the error and whether the interval covers it.
    # TODO: implement
    raise NotImplementedError

## Helper policy and self-check

In [ ]:
class _EnginePolicy:                       # same as game_manager.EnginePolicy, without the %run
    def __init__(self, e): self.e = e

    def choose_move(self, b):
        # TODO: delegate to the engine's play()
        raise NotImplementedError

# Self-check plan (write these as asserts once the functions above work):
# - Short games with depth-1 engines: three 16-ply games solid persona vs _EnginePolicy each have
#   16 moves.
# - evaluate_model_on_games on that history: 3 rows, the first OOD score is inf (no history yet),
#   model has 3 games afterwards.
# - One sloppy-persona game with update=False: finite OOD score, model still has 3 games.
# - Until style_move is written, a styled persona raises NotImplementedError.
# Example shape of a test:
# fast = MaterialEngine(depth=1)
# g = play_persona_game(PersonaPlayer(PERSONAS["solid"], fast, seed=1), _EnginePolicy(fast), max_plies=16)
# assert len(list(g.mainline_moves())) == 16
print("Self-check: not written yet (see plan above)")